# Tabla de predictores y puertas P2 y P3 — modelo neonatal NCHS

Etapas del framework: **E7** (predictores), puerta **P2** (fuga) y puerta **P3** (tamaño de muestra). Construye la tabla de contrato `data/contract/neonatal_nchs_v1/cohort_<año>.parquet` (no versionada) con los predictores conocidos al nacer + 5 minutos (ficha del desenlace, secciones 3 y 4), aplicando las decisiones registradas después de la auditoría de conteos el 23/11/2026:
- **M4-D1:** la sensibilidad (a) es un punto de referencia a las 48 horas (`at_risk_48h`, `label_day2_27`).
- **M4-D2:** se excluye el estado civil.
- **M4-D3:** desconocido es faltante, sin indicadores de faltante; el modelo principal lo imputa. Las marcas de imputación de NCHS de la edad gestacional y el peso se guardan en la tabla solo para verificar la calidad de los datos, no como predictores.

La puerta P3 sigue el framework 1.2: planeación con un R² supuesto del 15% de su máximo y luego reevaluación con el R² observado fuera de muestra, para el modelo principal y para el modelo de referencia (edad gestacional + peso al nacer). Salida agregada: `data/outputs_03_predictors_g2_g3.txt`. Versión en inglés: `03_predictors_g2_g3.ipynb` (mismo código). Tarda unos 10 minutos la primera vez y usa unos 6 GB de memoria.

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.nchs_reader import COHORTS, OUT_DIR

CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "neonatal_nchs_v1"))
OUT_FILE = ROOT / "data" / "outputs_03_predictors_g2_g3.txt"
DEV = [y for y in COHORTS if y <= 2021]          # temporal test: cohorts 2022-2023 (criterion C4)
SEED = 20261123
lines = []

def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)

def yn(s):
    """Y -> 1, N -> 0, U or blank -> missing (decision M4-D3: no missingness indicators)."""
    s = s.astype("string")
    return pd.Series(np.where(s.eq("Y").fillna(False), 1.0, np.where(s.eq("N").fillna(False), 0.0, np.nan)),
                     index=s.index)

def coded(s, unknown):
    s = s.astype("Float64").astype(float)
    return s.where(~s.isin(unknown))

def dummies(X, s, levels):
    """One column per non-reference level; missing stays missing in every column."""
    for name, codes in levels:
        X[name] = np.where(s.isna(), np.nan, s.isin(codes).astype(float))

def make_predictors(d):
    """Predictors known at birth + 5 minutes (outcome card, sections 3 and 4; decisions M4-D2, M4-D3).
    Reference levels: White, high school or less, private insurance, no earlier living birth, first
    pregnancy, care in the first trimester, cephalic presentation, spontaneous vaginal delivery,
    hospital birth, physician attendant, singleton."""
    X = pd.DataFrame(index=d.index)
    # mother (marital status excluded: M4-D2)
    X["age"] = coded(d["MAGER"], [])
    dummies(X, coded(d["MRACEHISP"], [8]), [("race_black", [2]), ("race_hispanic", [7]), ("race_asian", [4]),
                                            ("race_other", [3, 5, 6])])
    dummies(X, coded(d["MEDUC"], [9]), [("educ_some_college", [4, 5]), ("educ_bachelor_plus", [6, 7, 8])])
    dummies(X, coded(d["MBSTATE_REC"], [3]), [("born_outside_us", [2])])
    dummies(X, coded(d["PAY_REC"], [9]), [("pay_medicaid", [1]), ("pay_self", [3]), ("pay_other", [4])])
    X["wic"] = yn(d["WIC"])
    # obstetric history
    dummies(X, coded(d["PRIORLIVE"], [99]), [("prior_living_1", [1]), ("prior_living_2plus", list(range(2, 99)))])
    X["prior_child_died"] = coded(d["PRIORDEAD"], [99]).ge(1).astype(float).where(coded(d["PRIORDEAD"], [99]).notna())
    X["prior_other_termination"] = coded(d["PRIORTERM"], [99]).ge(1).astype(float).where(coded(d["PRIORTERM"], [99]).notna())
    ilp = coded(d["ILP_R"], [999])
    ilp = ilp.where(~ilp.between(0, 3))           # 000-003: plural-delivery code, not an interval
    first = ilp == 888                            # first pregnancy: reference level, not unknown
    for name, lo, hi in [("interval_lt18", 4, 17), ("interval_18_59", 18, 59), ("interval_60plus", 60, 300)]:
        X[name] = np.where(ilp.isna(), np.nan, (~first & ilp.between(lo, hi)).astype(float))
    X["prior_cesarean"] = yn(d["RF_CESAR"])
    X["prior_preterm_birth"] = yn(d["RF_PPB"])
    # current pregnancy
    pc = coded(d["PRECARE"], [99])
    X["care_after_first_trimester"] = np.where(pc.isna(), np.nan, pc.between(4, 10).astype(float))
    X["no_prenatal_care"] = np.where(pc.isna(), np.nan, (pc == 0).astype(float))
    X["prenatal_visits"] = coded(d["PREVIS"], [99])
    bmi = d["BMI"].astype("Float64").astype(float)
    X["bmi"] = bmi.where(bmi.between(12, 80))
    X["weight_gain"] = coded(d["WTGAIN"], [99])
    X["smoked_during_pregnancy"] = yn(d["CIG_REC"])
    for name, col in [("pregestational_diabetes", "RF_PDIAB"), ("gestational_diabetes", "RF_GDIAB"),
                      ("chronic_hypertension", "RF_PHYPE"), ("gestational_hypertension", "RF_GHYPE"),
                      ("eclampsia", "RF_EHYPE"), ("infertility_treatment", "RF_INFT"),
                      ("gonorrhea", "IP_GON"), ("syphilis", "IP_SYPH"), ("chlamydia", "IP_CHLAM"),
                      ("hepatitis_b", "IP_HEPB"), ("hepatitis_c", "IP_HEPC")]:
        X[name] = yn(d[col])
    # labor and delivery
    for name, col in [("induction", "LD_INDL"), ("augmentation", "LD_AUGM"), ("antenatal_steroids", "LD_STER"),
                      ("antibiotics_in_labor", "LD_ANTB"), ("chorioamnionitis", "LD_CHOR"),
                      ("anesthesia", "LD_ANES")]:
        X[name] = yn(d[col])
    dummies(X, coded(d["ME_PRES"], [9]), [("presentation_breech", [2]), ("presentation_other", [3])])
    dummies(X, coded(d["ME_ROUT"], [9]), [("delivery_forceps", [2]), ("delivery_vacuum", [3]),
                                          ("delivery_cesarean", [4])])
    tr = d["ME_TRIAL"].astype("string")
    X["trial_of_labor"] = np.where(tr.eq("Y").fillna(False), 1.0,
                                   np.where(tr.isin(["N", "X"]).fillna(False), 0.0, np.nan))
    dummies(X, coded(d["BFACIL3"], [3, 9]), [("born_out_of_hospital", [2])])
    dummies(X, coded(d["ATTEND"], [9]), [("attendant_midwife", [3, 4]), ("attendant_other", [5])])
    # newborn
    dummies(X, coded(d["DPLURAL"], []), [("twin", [2]), ("triplet_or_more", [3, 4, 5])])
    X["male"] = (d["SEX"].astype("string") == "M").astype(float)
    X["gestational_age"] = coded(d["COMBGEST"], [99])
    X["birthweight"] = coded(d["BRTHWGT"], [9999])
    X["apgar5"] = coded(d["APGAR5"], [99])
    for name, col in [("anencephaly", "CA_ANEN"), ("spina_bifida", "CA_MNSB"), ("cyanotic_heart_disease", "CA_CCHD"),
                      ("diaphragmatic_hernia", "CA_CDH"), ("omphalocele", "CA_OMPH"), ("gastroschisis", "CA_GAST"),
                      ("limb_reduction", "CA_LIMB"), ("cleft_lip", "CA_CLEFT"), ("cleft_palate", "CA_CLPAL"),
                      ("hypospadias", "CA_HYPO")]:
        X[name] = yn(d[col])
    for name, col in [("down_syndrome", "CA_DOWN"), ("chromosomal_disorder", "CA_DISOR")]:
        s = d[col].astype("string")       # C confirmed, P pending, N no, U unknown
        X[name] = np.where(s.isin(["C", "P"]).fillna(False), 1.0, np.where(s.eq("N").fillna(False), 0.0, np.nan))
    return X.astype("float32")

REFERENCE = ["gestational_age", "birthweight"]       # reference model fixed in the outcome card
PREDICTORS = list(make_predictors(pd.read_parquet(OUT_DIR / "cohort_2023.parquet").head(5)).columns)
print(len(PREDICTORS), "parameters:", PREDICTORS)

71 parameters: ['age', 'race_black', 'race_hispanic', 'race_asian', 'race_other', 'educ_some_college', 'educ_bachelor_plus', 'born_outside_us', 'pay_medicaid', 'pay_self', 'pay_other', 'wic', 'prior_living_1', 'prior_living_2plus', 'prior_child_died', 'prior_other_termination', 'interval_lt18', 'interval_18_59', 'interval_60plus', 'prior_cesarean', 'prior_preterm_birth', 'care_after_first_trimester', 'no_prenatal_care', 'prenatal_visits', 'bmi', 'weight_gain', 'smoked_during_pregnancy', 'pregestational_diabetes', 'gestational_diabetes', 'chronic_hypertension', 'gestational_hypertension', 'eclampsia', 'infertility_treatment', 'gonorrhea', 'syphilis', 'chlamydia', 'hepatitis_b', 'hepatitis_c', 'induction', 'augmentation', 'antenatal_steroids', 'antibiotics_in_labor', 'chorioamnionitis', 'anesthesia', 'presentation_breech', 'presentation_other', 'delivery_forceps', 'delivery_vacuum', 'delivery_cesarean', 'trial_of_labor', 'born_out_of_hospital', 'attendant_midwife', 'attendant_other', 'tw

## 1. Tabla de contrato y cobertura de los predictores (toda la población)

In [2]:
# Contract table, one file per cohort (not versioned: derived from NCHS data)
REBUILD = False
CONTRACT_DIR.mkdir(parents=True, exist_ok=True)
FIELDS = ["MAGER", "MRACEHISP", "MEDUC", "MBSTATE_REC", "PAY_REC", "WIC", "PRIORLIVE", "PRIORDEAD", "PRIORTERM",
          "ILP_R", "RF_CESAR", "RF_PPB", "PRECARE", "PREVIS", "BMI", "WTGAIN", "CIG_REC", "RF_PDIAB", "RF_GDIAB",
          "RF_PHYPE", "RF_GHYPE", "RF_EHYPE", "RF_INFT", "IP_GON", "IP_SYPH", "IP_CHLAM", "IP_HEPB", "IP_HEPC",
          "LD_INDL", "LD_AUGM", "LD_STER", "LD_ANTB", "LD_CHOR", "LD_ANES", "ME_PRES", "ME_ROUT", "ME_TRIAL",
          "BFACIL3", "ATTEND", "DPLURAL", "SEX", "COMBGEST", "BRTHWGT", "APGAR5", "CA_ANEN", "CA_MNSB", "CA_CCHD",
          "CA_CDH", "CA_OMPH", "CA_GAST", "CA_LIMB", "CA_CLEFT", "CA_CLPAL", "CA_HYPO", "CA_DOWN", "CA_DISOR"]
for y in COHORTS:
    final = CONTRACT_DIR / f"cohort_{y}.parquet"
    if final.exists() and not REBUILD:
        continue
    d = pd.read_parquet(OUT_DIR / f"cohort_{y}.parquet",
                        columns=["unit_id", "group_id", "RESTATUS", "COMBGST_IMP", "BWTIMP", "label_main",
                                 "label_early_neonatal", "d_age_days"] + [f for f in FIELDS if f != "COMBGEST"] + ["COMBGEST"])
    d = d[(d["RESTATUS"] < 4) & d["COMBGEST"].between(22, 47)].reset_index(drop=True)
    X = make_predictors(d)
    c = pd.DataFrame({"unit_id": d["unit_id"], "group_id": d["group_id"], "period": np.int16(y)})
    c["label_main"] = d["label_main"].to_numpy(bool)
    died_day0_1 = (c["label_main"] & d["d_age_days"].le(1).fillna(False)).to_numpy(bool)
    c["at_risk_48h"] = ~died_day0_1                                     # sensitivity (a), M4-D1
    c["label_day2_27"] = c["label_main"] & c["at_risk_48h"]
    c["label_early_neonatal"] = d["label_early_neonatal"].to_numpy(bool)  # sensitivity (b)
    c["ge28"] = (d["COMBGEST"] >= 28).to_numpy(bool)                     # sensitivity (c)
    c["ga_imputed"] = (d["COMBGST_IMP"].astype("string") == "1").fillna(False).to_numpy(bool)   # data quality only
    c["bw_imputed"] = (d["BWTIMP"].astype("string") == "1").fillna(False).to_numpy(bool)        # data quality only
    c = pd.concat([c, X], axis=1)
    c["n_unknown"] = X.isna().sum(axis=1).astype("int8")      # leakage check only, never a predictor
    tmp = final.with_suffix(".parquet.tmp")
    c.to_parquet(tmp, index=False); tmp.replace(final)
    print(f"cohort {y}: {len(c)} births written")
    del d, X, c

rows, n_total = {}, 0
for y in COHORTS:
    X = pd.read_parquet(CONTRACT_DIR / f"cohort_{y}.parquet", columns=PREDICTORS)
    n_total += len(X)
    for p in PREDICTORS:
        r = rows.setdefault(p, [0, 0.0, 0])
        r[0] += int(X[p].isna().sum()); r[1] += float(X[p].sum()); r[2] += int(X[p].notna().sum())
    del X
cov = pd.DataFrame({"pct_missing": {p: round(100 * rows[p][0] / n_total, 2) for p in PREDICTORS},
                    "mean_or_pct_yes": {p: round(rows[p][1] / rows[p][2] * (1 if p in
                        ["age", "prenatal_visits", "bmi", "weight_gain", "gestational_age", "birthweight", "apgar5"] else 100), 2)
                        for p in PREDICTORS}})
show(f"Q-P1 Contract table: {n_total} births, {len(PREDICTORS)} parameters; % missing and mean (continuous) or % yes, whole population", cov)

cohort 2016: 3938288 births written
cohort 2017: 3848558 births written
cohort 2018: 3785243 births written
cohort 2019: 3741187 births written
cohort 2020: 3607479 births written
cohort 2021: 3657762 births written
cohort 2022: 3661524 births written
cohort 2023: 3589826 births written

== Q-P1 Contract table: 29829867 births, 71 parameters; % missing and mean (continuous) or % yes, whole population
                            pct_missing  mean_or_pct_yes
age                                0.00            29.17
race_black                         0.91            14.40
race_hispanic                      0.91            24.41
race_asian                         0.91             6.26
race_other                         0.91             3.31
educ_some_college                  1.44            27.69
educ_bachelor_plus                 1.44            34.01
born_outside_us                    0.20            22.64
pay_medicaid                       0.68            41.98
pay_self                  

## 2. Puerta P2: verificaciones de diseño y prueba de fuga del número de desconocidos

In [3]:
# Gate G2 (leakage): design checks
EXCLUDED = ["label_main", "label_day2_27", "label_early_neonatal", "at_risk_48h", "ge28", "n_unknown",
            "ga_imputed", "bw_imputed", "marital_status",
            # recorded after the prediction point or reflecting the newborn's course (outcome card, section 3)
            "assisted_ventilation", "ventilation_6h", "nicu_admission", "surfactant", "newborn_antibiotics",
            "seizures", "apgar10", "transferred", "alive_at_report", "breastfed", "maternal_transfusion",
            "perineal_laceration", "uterine_rupture", "hysterectomy", "maternal_icu"]
checks = {
    "no outcome, landmark, death-certificate or post-prediction field among the predictors": not set(PREDICTORS) & set(EXCLUDED),
    "no missingness or imputation indicator among the predictors (M4-D3)":
        not any(k in p for p in PREDICTORS for k in ("missing", "unknown", "imputed")),
    "marital status excluded (M4-D2)": not any("marital" in p or "dmar" in p.lower() for p in PREDICTORS),
    "unit of analysis is the live birth and the group is the unit itself (no mother identifier)": True,
}
show("Gate G2: design checks", pd.Series(checks))
assert all(checks.values())

# G2 leakage check: AUROC of the number of unknown predictor fields alone (development cohorts)
def auroc_from_counts(neg, pos):
    """Exact AUROC (ties count one half) of an ordinal score from counts per score value."""
    neg, pos = np.asarray(neg, float), np.asarray(pos, float)
    below = np.concatenate([[0], np.cumsum(neg)[:-1]])
    return float(((below + neg / 2) * pos).sum() / (neg.sum() * pos.sum()))

res = {}
for name, cond in [("population", None), ("ge28_weeks", "ge28")]:
    tab = None
    for y in DEV:
        c = pd.read_parquet(CONTRACT_DIR / f"cohort_{y}.parquet", columns=["n_unknown", "label_main", "ge28"])
        if cond:
            c = c[c[cond]]
        t = pd.crosstab(c["n_unknown"], c["label_main"])
        tab = t if tab is None else tab.add(t, fill_value=0)
    tab = tab.reindex(columns=[False, True], fill_value=0).astype(np.int64)
    res[name] = round(auroc_from_counts(tab[False], tab[True]), 3)
    if name == "population":
        show("G2 leakage check: births by number of unknown predictor fields (development cohorts)",
             pd.DataFrame({"births": tab.sum(axis=1), "pct": (100 * tab.sum(axis=1) / tab.values.sum()).round(2)}))
show("G2 leakage check: AUROC of the number of unknown fields alone (development cohorts)", pd.Series(res))


== Gate G2: design checks
no outcome, landmark, death-certificate or post-prediction field among the predictors         True
no missingness or imputation indicator among the predictors (M4-D3)                           True
marital status excluded (M4-D2)                                                               True
unit of analysis is the live birth and the group is the unit itself (no mother identifier)    True

== G2 leakage check: births by number of unknown predictor fields (development cohorts)
             births    pct
n_unknown                 
0          17911137  79.33
1            555683   2.46
2            492097   2.18
3           2748733  12.17
4            257239   1.14
5            200392   0.89
6            195772   0.87
7             58914   0.26
8             46622   0.21
9             26835   0.12
10            15683   0.07
11             9384   0.04
12            20266   0.09
13             7564   0.03
14             5257   0.02
15             6011   0.03
16

## 3. Puerta P3: planeación y reevaluación (cohortes de desarrollo 2016–2021)

In [4]:
# Gate G3 (sample size), two steps (framework 1.2): planning with an assumed Cox-Snell R2 of 15% of its
# maximum, then reassessment with the observed out-of-fold R2 of a cross-validated logistic regression.
# Development cohorts only. All events and a random fraction of non-events, weighted, so that the
# log-likelihood estimates that of the full data. Parameters = one-hot columns, without the intercept.
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

FRACTION = 0.03
SETS = {"main model (all predictors)": PREDICTORS, "reference model (gestational age + birthweight)": REFERENCE}
rng = np.random.default_rng(SEED)
parts, n_dev, e_dev = [], 0, 0
for y in DEV:
    c = pd.read_parquet(CONTRACT_DIR / f"cohort_{y}.parquet", columns=PREDICTORS + ["label_main"])
    n_dev += len(c); e_dev += int(c["label_main"].sum())
    keep = c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)
    parts.append(c[keep]); del c
s = pd.concat(parts, ignore_index=True); del parts
y_s = s.pop("label_main").to_numpy(bool)
w_s = np.where(y_s, 1.0, 1.0 / FRACTION)
phi = e_dev / n_dev
L0n = phi * np.log(phi) + (1 - phi) * np.log(1 - phi)
R2max = 1 - np.exp(2 * L0n)

def riley(P, r2, n_avail, shrink=0.9):
    n1 = P / ((shrink - 1) * np.log(1 - r2 / shrink))
    s2 = r2 / (r2 + 0.05 * R2max)
    n2 = P / ((s2 - 1) * np.log(1 - r2 / s2))
    n3 = (1.96 / 0.05) ** 2 * phi * (1 - phi)
    req = int(np.ceil(max(n1, n2, n3)))
    smax = max(shrink, s2)
    return {"n required (shrinkage)": int(np.ceil(n1)), "n required (optimism)": int(np.ceil(n2)),
            "n required (overall risk)": int(np.ceil(n3)), "n required": req,
            "events required": int(np.ceil(req * phi)),
            "parameters supported": int(np.floor(n_avail * (smax - 1) * np.log(1 - r2 / smax))),
            "gate G3": "PASSED" if n_avail >= req and e_dev >= req * phi else "NOT PASSED"}

out = {}
for name, cols in SETS.items():
    P = len(cols)
    plan = riley(P, 0.15 * R2max, n_dev)
    oof = np.zeros(len(s))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(s, y_s):
        m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=1e6, max_iter=2000))
        m.fit(s.iloc[tr][cols], y_s[tr], logisticregression__sample_weight=w_s[tr])
        oof[te] = m.predict_proba(s.iloc[te][cols])[:, 1]
    p = np.clip(oof, 1e-12, 1 - 1e-12)
    n = float(w_s.sum())
    L1 = float(np.sum(w_s * (y_s * np.log(p) + (~y_s) * np.log(1 - p))))
    R2 = 1 - np.exp(2 * (n * L0n - L1) / n)
    obs = riley(P, R2, n_dev)
    out[name] = {"candidate parameters": P, "events per parameter": round(e_dev / P, 1),
                 "assumed R2 (15% of max)": round(0.15 * R2max, 5),
                 **{f"planning: {k}": v for k, v in plan.items()},
                 "observed Cox-Snell R2 (cross-validated)": round(R2, 5), "% of maximum": round(100 * R2 / R2max, 1),
                 **{f"reassessment: {k}": v for k, v in obs.items()}}
show("Gate G3: data (development cohorts 2016-2021)",
     pd.Series({"development births": n_dev, "development events": e_dev, "prevalence": round(phi, 5),
                "Cox-Snell R2 max": round(R2max, 5), "sample used (all events + 3% of non-events)": len(s)}, dtype=object))
show("Gate G3: sample size (Riley et al.), planning and reassessment", pd.DataFrame(out))
show("Primary learner rule (events per parameter >= 20: best base learner of a preliminary run)",
     "events per parameter = %.1f" % (e_dev / len(PREDICTORS)))

OUT_FILE.write_text("Predictor table and gates G2 and G3 of the neonatal model (aggregate output only)\n"
                    + "\n".join(lines) + "\n", encoding="utf-8")
print("written:", OUT_FILE)


== Gate G3: data (development cohorts 2016-2021)
development births                             22578517
development events                                62713
prevalence                                      0.00278
Cox-Snell R2 max                                0.03752
sample used (all events + 3% of non-events)      738384

== Gate G3: sample size (Riley et al.), planning and reassessment
                                        main model (all predictors) reference model (gestational age + birthweight)
candidate parameters                                             71                                               2
events per parameter                                          883.3                                         31356.5
assumed R2 (15% of max)                                     0.00563                                         0.00563
planning: n required (shrinkage)                             113174                                            3188
planning: n required (o